In [ ]:
from sklearn import datasets
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import math


In [ ]:
mean1 = [-1 for _ in range(100)]
mean2 = [1 for _ in range(100)]


In [ ]:
mean1

In [ ]:
mean2

In [ ]:
cov_1 = [0.5 for _ in range(100)]
cov_mat_1 = np.diag(cov_1)
cov_mat_1

array([[0.5, 0. , 0. , ..., 0. , 0. , 0. ],
       [0. , 0.5, 0. , ..., 0. , 0. , 0. ],
       [0. , 0. , 0.5, ..., 0. , 0. , 0. ],
       ...,
       [0. , 0. , 0. , ..., 0.5, 0. , 0. ],
       [0. , 0. , 0. , ..., 0. , 0.5, 0. ],
       [0. , 0. , 0. , ..., 0. , 0. , 0.5]])

In [ ]:
A1_1 = np.random.multivariate_normal(mean1, cov_mat_1 ,20)  #normal
A2_1 = np.random.multivariate_normal(mean2, cov_mat_1 ,20) 
A3_1 = np.random.multivariate_normal(mean1, cov_mat_1 ,20) 

In [ ]:
A1_1

array([[-1.1766366 , -0.80576526, -0.74942944, ..., -0.48402987,
        -0.89408163, -0.53208245],
       [-0.55732113, -1.75494848, -1.71728129, ..., -0.41481287,
        -0.93113778, -0.8888336 ],
       [ 0.05270929,  0.01972448, -0.86203601, ..., -1.59607668,
        -1.3519807 , -0.77943064],
       ...,
       [-2.2464988 , -0.23358642, -1.72406499, ..., -2.22616052,
        -0.97966493,  0.15300218],
       [-1.02847414, -0.78854335, -1.26451078, ..., -1.91228108,
        -0.75196132, -1.61295142],
       [-1.92997349, -1.53728114, -1.34128362, ...,  0.76460115,
        -0.58020356, -1.09853374]])

In [ ]:
l_1 = [f'f{i}' for i in range(1, 101)]
df_A1_1 = pd.DataFrame(A1_1, columns = l_1)
df_A2_1 = pd.DataFrame(A2_1, columns = l_1)
df_A3_1 = pd.DataFrame(A3_1, columns = l_1)


df_1 = pd.concat([df_A1_1, df_A2_1, df_A3_1], axis = 0, ignore_index = True)
df_1

In [ ]:
X_1 = df_1.to_numpy(dtype = None, copy = False)

In [ ]:
dist_mat_1 = np.zeros((len(X_1),len(X_1)))

In [ ]:
for i in range(1, len(dist_mat_1[0])):
  for j in range(i):
    for k in range(len(dist_mat_1[0])):
      if k == i or k == j:
        continue
        print(i)
      dist_mat_1[i][j] += abs(np.linalg.norm(X_1[i] - X_1[k]) - np.linalg.norm(X_1[j] - X_1[k]))
    dist_mat_1[i][j] /= (len(dist_mat_1[0])-2)

In [ ]:
def ret_min(a, b): # Returns minimum of a and b
  if a>b:
    return b

  return a

In [ ]:
def clustering(matrix, hierarchy):
  
  # Base case
  if len(matrix) == 2:
    min_i = 0
    min_j = 1
    hierarchy = hierarchy[:min_i] + [hierarchy[min_i], hierarchy[min_j]] + hierarchy[min_j+1:]  
    return hierarchy

  min = matrix[1][0] #initializing minimum distance
  min_i, min_j = 1, 0

  for i in range(1, len(matrix[0])):
    for j in range(i):
      if i>j and ((i-j) == 1):
        if matrix[i][j]<min:
          min = matrix[i][j]
          min_i = i
          min_j = j
  
  '''
  b = np.where(matrix == 0.0, max, matrix)
  min = np.min(b)
  min_i, min_j = np.where(b == np.min(b))
  min_i, min_j = min_i[0], min_j[0]
  
  '''
  

  # Swapping min_i, min_j in order to maintain min_i < min_j
  if min_i > min_j:
    a = min_i
    min_i = min_j
    min_j = a
  
  
  hierarchy = hierarchy[:min_i] + [[hierarchy[min_i], hierarchy[min_j]]] + hierarchy[min_j+1:]

  dist_mat = np.copy(matrix) # Copying the original matrix so that original matrix is not affected by further operations
  dist_mat = np.delete(dist_mat, (min_j), axis=0) # Deleting  min_j row
  dist_mat = np.delete(dist_mat, (min_j), axis=1) # Deleting  min_j column

  """
  # Copying the remaining matrix to new distance matrix for recursion
  dist_mat_2 = np.zeros((len(matrix)-1, len(matrix)-1))
  for i in range(len(matrix)-2):
    for j in range(len(matrix)-2):
      dist_mat_2[i][j] = dist_mat[i][j]
  """

  # Deleting dist_mat in order to save space
  #dist_mat = np.delete(dist_mat, [i for i in range(len(dist_mat))], 0)

  dist_min_i = [] # List of distances from point min_i to all other points except min_j
  dist_min_j = [] # List of distances from point min_j to all other points except min_i

  for i in range(len(matrix)):
    if i<min_i:
      dist_min_i += [matrix[min_i][i]]
      dist_min_j += [matrix[min_j][i]]
    elif i == min_i or i == min_j:
      continue
    elif i<min_j:
      dist_min_i += [matrix[i][min_i]]
      dist_min_j += [matrix[min_j][i]]
    else:
      dist_min_i += [matrix[i][min_i]]
      dist_min_j += [matrix[i][min_j]]
  
  # Filling the last row of new distance matrix
  for i in range(len(dist_min_i)):
    if i < min_i:
      dist_mat[min_i][i] = ret_min(dist_min_i[i], dist_min_j[i])
    else:
      dist_mat[i+1][min_i] = ret_min(dist_min_i[i], dist_min_j[i])
  
  dist_min_i = []
  dist_min_j = []

  if (len(hierarchy) == 3):
    return hierarchy
  '''
  t += 1
  if t == 5:
    return hierarchy  
  '''
  hierarchy = clustering(dist_mat, hierarchy) # The recusion call

  return hierarchy

In [ ]:
hier1 = [[i] for i in range(len(dist_mat_1))]
cluster_first = clustering(dist_mat_1, hier1)
print(hier1)

[[0], [1], [2], [3], [4], [5], [6], [7], [8], [9], [10], [11], [12], [13], [14], [15], [16], [17], [18], [19], [20], [21], [22], [23], [24], [25], [26], [27], [28], [29], [30], [31], [32], [33], [34], [35], [36], [37], [38], [39], [40], [41], [42], [43], [44], [45], [46], [47], [48], [49], [50], [51], [52], [53], [54], [55], [56], [57], [58], [59]]


In [ ]:
print(cluster_first[0])
print(cluster_first[1])
print(cluster_first[2])

[[[[[0], [1]], [2]], [3]], [[4], [[5], [[6], [[7], [[8], [[[[[[9], [10]], [11]], [[12], [[13], [[14], [[15], [16]]]]]], [[17], [18]]], [19]]]]]]]]
[[[20], [21]], [[22], [[23], [[[24], [25]], [[26], [[27], [[28], [[[[[[[[29], [30]], [31]], [32]], [33]], [[34], [[35], [[36], [37]]]]], [38]], [39]]]]]]]]]
[[[[40], [41]], [42]], [[43], [[44], [[45], [[46], [[47], [[[[[[[[[48], [49]], [[50], [51]]], [52]], [53]], [54]], [[55], [[56], [57]]]], [58]], [59]]]]]]]]


In [ ]:
from collections import Iterable

def flatten(lis):
     for item in lis:
         if isinstance(item, Iterable) and not isinstance(item, str):
             for x in flatten(item):
                 yield x
         else:        
             yield item

<ipython-input-16-3b6c95042051>:1: DeprecationWarning: Using or importing the ABCs from 'collections' instead of from 'collections.abc' is deprecated since Python 3.3, and in 3.10 it will stop working
  from collections import Iterable


In [ ]:
cluster_first_1 = sorted(list(flatten(cluster_first[1])))
cluster_first_2 = sorted(list(flatten(cluster_first[0])))
cluster_first_3 = sorted(list(flatten(cluster_first[2])))

In [ ]:
print(cluster_first_1)
print(cluster_first_2)
print(cluster_first_3)

[20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39]
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
[40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59]
